# What Are Generative Models?

This notebook accompanies the **ML Viz** lesson on generative models.
We'll explore the difference between discriminative and generative approaches,
and visualize what it means to learn a data distribution.

**Companion lesson:** https://ml-viz.vercel.app/courses/generative-models/01-what-are-generative-models

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## Discriminative vs Generative

- **Discriminative models** learn $P(y \mid x)$ — the boundary between classes.
- **Generative models** learn $P(x)$ — the distribution of the data itself.

Let's see the difference on a 2D toy dataset.

In [ ]:
np.random.seed(42)

# Two Gaussian clusters (class 0 and class 1)
n = 200
X_class0 = np.random.randn(n, 2) * 0.8 + np.array([-1.5, 0])
X_class1 = np.random.randn(n, 2) * 0.8 + np.array([1.5, 0])
X = np.vstack([X_class0, X_class1])
y = np.array([0] * n + [1] * n)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: data with labels
axes[0].scatter(X_class0[:, 0], X_class0[:, 1], c='#f43f5e', s=15, alpha=0.7, label='Class 0')
axes[0].scatter(X_class1[:, 0], X_class1[:, 1], c='#818cf8', s=15, alpha=0.7, label='Class 1')
axes[0].set_title('What a Discriminative Model Sees', color='white', fontsize=12)
axes[0].legend(fontsize=9)

# Right: just the data distribution
axes[1].scatter(X[:, 0], X[:, 1], c='#94a3b8', s=15, alpha=0.5)
axes[1].set_title('What a Generative Model Sees', color='white', fontsize=12)

for ax in axes:
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

plt.tight_layout()
plt.show()

## Learning $P(x)$: the density function

A generative model learns the underlying probability density. For our two-cluster
data, the density is a mixture of two Gaussians:

$$P(x) = \pi_0 \cdot \mathcal{N}(x \mid \mu_0, \Sigma_0) + \pi_1 \cdot \mathcal{N}(x \mid \mu_1, \Sigma_1)$$

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# 2D density heatmap using histogram
h = ax.hist2d(X[:, 0], X[:, 1], bins=40, cmap='magma', density=True)
plt.colorbar(h[3], ax=ax, label='Density')

ax.set_title('Learned Data Density $P(x)$', color='white', fontsize=13)
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
plt.tight_layout()
plt.show()

print('A generative model learns this density surface.')
print('Once learned, we can sample from it to create new, realistic data points.')

## Generating new samples

Once we know $P(x)$, we can **sample** from it to create entirely new data points
that look like the original data. Let's do this with our Gaussian mixture.

In [ ]:
# "Generative model" = we know the parameters (learned via EM or similar)
pi_0, pi_1 = 0.5, 0.5
mu_0, mu_1 = X_class0.mean(axis=0), X_class1.mean(axis=0)
cov_0 = np.cov(X_class0.T)
cov_1 = np.cov(X_class1.T)

# Generate new samples
n_gen = 200
labels = np.random.choice([0, 1], size=n_gen, p=[pi_0, pi_1])
X_generated = np.array([
    np.random.multivariate_normal(mu_0, cov_0) if l == 0 else np.random.multivariate_normal(mu_1, cov_1)
    for l in labels
])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(X[:, 0], X[:, 1], c='#94a3b8', s=15, alpha=0.4)
axes[0].set_title('Original Data', color='white', fontsize=12)

axes[1].scatter(X_generated[:, 0], X_generated[:, 1], c='#14b8a6', s=15, alpha=0.6)
axes[1].set_title('Generated Samples', color='white', fontsize=12)

for ax in axes:
    ax.set_xlim(X[:, 0].min() - 1, X[:, 0].max() + 1)
    ax.set_ylim(X[:, 1].min() - 1, X[:, 1].max() + 1)
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

plt.tight_layout()
plt.show()

## The latent space idea

Most deep generative models use a **latent space** — a compressed representation.
The workflow is:

1. Sample a point $z$ from a simple distribution (e.g., $\mathcal{N}(0, I)$)
2. Pass it through a **decoder** network to produce a data point $x$

Let's simulate this with a simple decoder on MNIST-like data.

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
fig.suptitle('Simulating Latent Space → Image Generation', color='white', fontsize=13, y=1.02)

# Create simple "digit-like" patterns from 2D latent space
z_vals = np.linspace(-2, 2, 8)
for i, z in enumerate(z_vals):
    # Simple pattern: concentric rings with varying radius based on z
    radius = 0.3 + 0.5 * (z + 2) / 4  # z controls size
    
    # Top row: one "digit" pattern
    ax = axes[0, i]
    theta = np.linspace(0, 2 * np.pi, 100)
    ax.plot(radius * np.cos(theta), radius * np.sin(theta), color='#818cf8', linewidth=2)
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.set_aspect('equal')
    ax.set_title(f'z={z:.1f}', color='#94a3b8', fontsize=8)
    ax.axis('off')
    
    # Bottom row: another pattern
    ax = axes[1, i]
    r2 = 0.2 + 0.3 * (z + 2) / 4
    ax.plot(r2 * np.cos(theta), r2 * np.sin(theta), color='#14b8a6', linewidth=2)
    ax.plot((r2 * 0.5) * np.cos(theta), (r2 * 0.5) * np.sin(theta), color='#f43f5e', linewidth=1.5)
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.set_aspect('equal')
    ax.axis('off')

plt.tight_layout()
plt.show()

print('As z changes smoothly, the generated output changes smoothly.')
print('This is the key property of a well-structured latent space.')

## Model families overview

| Family | How it models $P(x)$ | Key insight |
|--------|----------------------|-------------|
| **VAE** | Approximate inference over latent space | Reparameterization trick makes it differentiable |
| **GAN** | Generator vs discriminator game | No explicit density — implicit sampling |
| **Diffusion** | Iteratively denoise | Forward process is easy; learn to reverse it |
| **Autoregressive** | Factor into conditionals | $P(x) = \prod P(x_i \mid x_{<i})$ |

We'll explore VAEs, GANs, and Diffusion in the following lessons.

In [ ]:
# Visual comparison: real vs generated from a simple generative model
np.random.seed(12)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
fig.suptitle('Generative Modeling Pipeline', color='white', fontsize=13, y=1.02)

# Step 1: Sample from latent space
z_samples = np.random.randn(50, 2)
axes[0].scatter(z_samples[:, 0], z_samples[:, 1], c='#818cf8', s=20, alpha=0.7)
circle = plt.Circle((0, 0), 2, fill=False, color='#818cf8', linestyle='--', alpha=0.5)
axes[0].add_patch(circle)
axes[0].set_title('1. Sample $z \sim N(0, I)$', color='white', fontsize=11)
axes[0].set_xlim(-3, 3)
axes[0].set_ylim(-3, 3)
axes[0].set_aspect('equal')

# Step 2: Decode
decoded = z_samples * np.array([1.2, 0.8]) + np.array([0.5, -0.3])  # simple transform
axes[1].scatter(decoded[:, 0], decoded[:, 1], c='#14b8a6', s=20, alpha=0.7)
axes[1].set_title('2. Decode $G(z) = x$', color='white', fontsize=11)
axes[1].set_xlim(-3, 3)
axes[1].set_ylim(-3, 3)
axes[1].set_aspect('equal')

# Step 3: Real vs Generated
axes[2].scatter(X[:, 0], X[:, 1], c='#94a3b8', s=15, alpha=0.4, label='Real')
axes[2].scatter(decoded[:, 0], decoded[:, 1], c='#eab308', s=20, alpha=0.7, label='Generated')
axes[2].set_title('3. Real vs Generated', color='white', fontsize=11)
axes[2].legend(fontsize=9)
axes[2].set_xlim(-3, 3)
axes[2].set_ylim(-3, 3)
axes[2].set_aspect('equal')

plt.tight_layout()
plt.show()

## Key takeaways

- **Discriminative** models learn $P(y\mid x)$ (boundaries); **generative** models learn $P(x)$ (the data itself).
- Generative models can **sample** new data, fill in missing values, and detect anomalies.
- Families: explicit density (VAE, autoregressive), implicit (GAN), and score/diffusion.
- A **latent space** is a compact code from which realistic samples are decoded.